<!-- cspell:ignore atleast dalitz mystnb -->

# Phase space generators

:::{autolink-concat}
:::

Amplitude analysis needs samples of four-momenta that are distributed uniformly over the phase space of a decay. In TensorWaves, such a sample comes from a {class}`.DataGenerator`. This page compares the available generators, so that you can decide which one fits your use case. All of them take the same masses and return a {obj}`.DataSample` with the same structure, so they can be swapped without changing the rest of an analysis.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
set_matplotlib_formats("svg")

## Available generators

Each generator gets its own {class}`.RealNumberGenerator`, because not every generator can work with every type of random number generator. The mapping below is the single place that defines which generators this page compares. All tables and figures on this page loop over it.

In [ ]:
from tensorwaves.data import (
    EvtGenPhaseSpaceGenerator,
    NumpyUniformRNG,
    TFPhaseSpaceGenerator,
    TFUniformRealNumberGenerator,
)

initial_state_mass = 3.0969
final_state_masses = {0: 0.1350, 1: 0.1396, 2: 0.4937}
tf_generator = TFPhaseSpaceGenerator(initial_state_mass, final_state_masses)
tf_generator.show_progress = False
generators = {
    "phasespace (TensorFlow)": (
        tf_generator,
        TFUniformRealNumberGenerator(seed=0),
    ),
    "EvtGen": (
        EvtGenPhaseSpaceGenerator(initial_state_mass, final_state_masses),
        NumpyUniformRNG(seed=0),
    ),
}

| Generator | Package | Extra | Events | {class}`.RealNumberGenerator` |
| --- | --- | --- | --- | --- |
| {class}`.TFPhaseSpaceGenerator` | [`phasespace`](https://phasespace.readthedocs.io) | `phsp` | unweighted through hit-and-miss | {class}`.TFUniformRealNumberGenerator` |
| {class}`.TFWeightedPhaseSpaceGenerator` | [`phasespace`](https://phasespace.readthedocs.io) | `phsp` | weighted | {class}`.TFUniformRealNumberGenerator` |
| {class}`.EvtGenPhaseSpaceGenerator` | [`pyevtgen`](https://github.com/redeboer/pyevtgen-test) | `evtgen` | unweighted | any |

The installation of these extras is described on {doc}`/install`.

## Generating a sample

Every generator has the same {meth}`~.DataGenerator.generate` method, which returns a {obj}`dict` of final state IDs to arrays of four-momenta $\left(E, p_x, p_y, p_z\right)$ of shape $n \times 4$. We also record how long each generator takes. These timings only give an impression for this three-body decay on the machine that builds this page.

In [ ]:
n_events = 100_000
samples = {}
timings = {}
for name, (generator, rng) in generators.items():
    start = time.perf_counter()
    samples[name] = generator.generate(n_events, rng)
    timings[name] = time.perf_counter() - start

In [ ]:
pd.DataFrame(
    {
        "Final state": [sorted(sample) for sample in samples.values()],
        "Shape": [sample["p0"].shape for sample in samples.values()],
        "Time [s]": [round(duration, 3) for duration in timings.values()],
    },
    index=list(samples),
)

## Comparing distributions

The events of different generators are not identical, not even with the same seed, but they should follow the same distribution. A uniform phase space sample shows up as a flat Dalitz plot within the kinematic boundary, and the invariant mass projections should agree between generators.

In [ ]:
def invariant_mass_squared(p: np.ndarray) -> np.ndarray:
    return p[:, 0] ** 2 - np.sum(p[:, 1:] ** 2, axis=1)


fig, axes = plt.subplots(
    ncols=len(samples), figsize=(4 * len(samples), 3.5), sharex=True, sharey=True
)
for ax, (name, sample) in zip(np.atleast_1d(axes), samples.items(), strict=True):
    s01 = invariant_mass_squared(sample["p0"] + sample["p1"])
    s12 = invariant_mass_squared(sample["p1"] + sample["p2"])
    ax.hist2d(s01, s12, bins=60, cmin=1, rasterized=True)
    ax.set_title(name)
    ax.set_xlabel("$m_{01}^2$ [GeV$^2$]")
np.atleast_1d(axes)[0].set_ylabel("$m_{12}^2$ [GeV$^2$]")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, sample in samples.items():
    m01 = np.sqrt(invariant_mass_squared(sample["p0"] + sample["p1"]))
    ax.hist(m01, bins=60, histtype="step", label=name)
ax.set_xlabel("$m_{01}$ [GeV]")
ax.set_ylabel("Events")
ax.legend()
fig.tight_layout()
plt.show()

## Which generator to choose

{class}`.TFPhaseSpaceGenerator` is the default choice and it is what the {doc}`/amplitude-analysis` page uses. Its {class}`.TFWeightedPhaseSpaceGenerator` variant provides event weights, which is useful for the {class}`.IntensityDistributionGenerator`.

{class}`.EvtGenPhaseSpaceGenerator` does not depend on TensorFlow and works with any {class}`.RealNumberGenerator`. The {class}`.RealNumberGenerator` only provides a seed for the random engine of EvtGen. There are two limitations:

- It provides no weights, because EvtGen applies accept-reject sampling internally.
- Its accept-reject efficiency drops by about a factor of ten for each additional final state particle, which makes it impractically slow beyond about nine particles. A running call cannot be interrupted.